In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
약효분류군별 성장 분석 파이프라인 v2  (YoY 기반 EI + 장단기 성장 + 리스크)
────────────────────────────────────────────────────────────────────────
입력 : meft_2020-2025.csv (월별 · 약효분류별 수량/사용금액)

핵심 지표
  · YoY_cls, YoY_mkt          : 전년 대비 성장률 (모든 EI의 기반)
  · EI_t  = (1+YoY_cls)/(1+YoY_mkt) × 100      (100 = 시장과 같은 속도)
  · EI_3Y = 최근 3개년 가중평균 EI (0.2 / 0.3 / 0.5)
  · 성장기여도(%p) = 분류군 증가액 ÷ 시작연도 전체시장 × 100  (합계 = 시장 성장률)
  · 세그먼트 = EI_3Y(≥100) × 점유율(MS, 중앙값 기준) 4분면

출력 (--out 폴더)
  1_matrix_A_promising.png     Matrix A : MS × 3Y 가중 EI  (유망성장군 발굴)
  2_matrix_B_drivers.png       Matrix B : 성장기여도 × 장기 CAGR (성장견인군)
  3_yoy_heatmap_pool.png       POOL(핵심주도+유망성장 전체) 연도별 YoY 히트맵 — 성장 패턴 일관성
  4_ei_trend_pool.png          POOL 전체 EI 추세(소형 다중) — 시장 대비 성장 지속성
  5_log_ttm_pool.png           POOL 전체 Log-TTM 매출 궤적(소형 다중) — 최근 성장세 유지/가속 여부
  A1~A3_*.png                  부록: 장·단기 CAGR 매트릭스 / 규모 상위 히트맵 / 성장 vs MDD
  growth_report.xlsx / .csv    통합 엑셀 리포트(지표·세그먼트·EI·YoY·차트 포함)

실행 : python atc_growth_analysis.py --csv meft_2020-2025.csv --out output
필요 : pandas, numpy, matplotlib, openpyxl (차트 삽입 시 pillow)
"""
import argparse
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore")


In [14]:

# ═════════════════════════════════════════════════════════════
# 0. 설정
# ═════════════════════════════════════════════════════════════
UNITS = {  # 지표 컬럼 : (억원 환산용 나눗수, 단위 라벨)
    "사용금액_백만원": (100, "억원"),
    "수량_천개": (1000, "백만개"),
}
EXCLUDE_CODES = [999]        # '해당사항없음' 등 분석 제외 코드
MAX_MISSING_MONTHS = 0       # 결측 월이 이 값보다 많은 분류군은 제외(상위 그룹 행 등)
ZERO_FLOOR = 0.1             # 최종 분석대상의 초기 0값(censored) 치환값 (지표 원단위)
ZERO_LEADING_ONLY = True     # True: 첫 양수 이전의 초기 0만 / False: 모든 0
MIN_RECENT = 500         # 최근연도 규모 하한 (백만원) — 미만 제거
MIN_BASE = 300                 # 단위; AMOUNT => (백만원), QUANTITY => (천개)
SHORT_WINDOW = 3             # 단기 CAGR 기간(년)
# ACCEL_TOL = 0.02             # 단기−장기 격차 ±2%p 이내면 '유지'
 # 2. tolerance 분리
ACCEL_TOL_CAGR = 0.02
ACCEL_TOL_YOY  = 0.05

EI_WEIGHTS = (0.2, 0.3, 0.5)  # (end-2, end-1, end) 연도 EI 가중치
DRIVER_PP = 0.3              # 성장견인군 판정: 성장기여도 하한(%p)

SEG_PROM = "유망성장군 (High EI, Low MS)"
SEG_CORE = "핵심주도군 (High EI, High MS)"
SEG_CASH = "캐시카우 (Low EI, High MS)"
SEG_DECL = "정체/쇠퇴군 (Low EI, Low MS)"
SEG_COLORS = {SEG_PROM: "#1b9e77", SEG_CORE: "#2b7bba", SEG_CASH: "#e6a117", SEG_DECL: "#c8515f"}
SEG_SHADE = {SEG_PROM: "#e6f5ef", SEG_CORE: "#e8f1fa", SEG_CASH: "#fdf3e4", SEG_DECL: "#f8eaea"}

C_UP, C_DOWN, C_FLAT, C_MKT = "#2b7bba", "#d1495b", "#9aa5b1", "#222222"


def setup_font():
    cands = ["Noto Sans CJK KR", "Noto Sans KR", "Malgun Gothic", "AppleGothic", "NanumGothic", "Nanum Gothic",
             "NanumBarunGothic",  "Noto Sans CJK JP"]
    installed = {f.name for f in fm.fontManager.ttflist}
    for c in cands:
        if c in installed:
            plt.rcParams["font.family"] = c
            break
    else:
        print("[경고] 한글 폰트를 찾지 못했습니다. NanumGothic 등을 설치하세요.")
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["axes.spines.top"] = False
    plt.rcParams["axes.spines.right"] = False


def short_name(s, n=11):
    s = str(s)
    return s if len(s) <= n else s[: n - 1] + "…"


def pct_fmt(v, _=None):
    return f"{v:.0f}%"


# ═════════════════════════════════════════════════════════════
# 1. 데이터 로드 · 정제
# ═════════════════════════════════════════════════════════════
def load(csv_path, metric):
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
    df["date"] = pd.to_datetime(df["진료년월"] + "-01")
    names = df.drop_duplicates("약효분류코드").set_index("약효분류코드")["약효분류명"]
    monthly = df.pivot(index="date", columns="약효분류코드", values=metric).sort_index()
    return monthly, names


def select_classes(monthly, start, end, min_base, min_recent):
    """분석 대상 선정: 결측 없음 + 기준연도 규모 + 최근연도 규모(0.5억 이상)."""
    # 방어코드
    n_months_start = monthly.index[monthly.index.year == start].nunique()
    if n_months_start < 12:
        raise ValueError(
            f"{start}년 데이터가 {n_months_start}개월뿐입니다. "
            f"--start {start+1} 로 실행하세요."
        )
    n_months_end = monthly.index[monthly.index.year == end].nunique()
    if n_months_end < 12:
        raise ValueError(
            f"{end}년 데이터가 {n_months_end}개월뿐입니다. "
            f"--end {end-1} 로 실행하거나 완전연도를 지정하세요."
        )
    # 계산코드
    n_missing = monthly.isna().sum()
    base = monthly[monthly.index.year == start].sum()
    recent = monthly[monthly.index.year == end].sum()
    reason = pd.Series("", index=monthly.columns)
    reason[n_missing > MAX_MISSING_MONTHS] = "결측월 많음(상위그룹/신규)"
    reason[(reason == "") & (recent < min_recent)] = "최근연도 규모 하한 미달"
    reason[(reason == "") & (base < min_base)] = "기준연도 규모 하한 미달"
    reason[reason.index.isin(EXCLUDE_CODES)] = "제외코드"
    return reason[reason == ""].index.tolist(), reason[reason != ""]


def apply_zero_floor(monthly, classes):
    """최종 분석대상에 한해 censored 0값을 ZERO_FLOOR로 치환."""
    monthly = monthly.copy()
    n_total = 0
    for c in classes:
        col = monthly[c]
        if ZERO_LEADING_ONLY:
            if col.gt(0).any():
                mask = (col == 0) & (col.index < col.gt(0).idxmax())
            else:
                mask = col == 0
        else:
            mask = col == 0
        if mask.any():
            monthly.loc[mask, c] = ZERO_FLOOR
            n_total += int(mask.sum())
    print(f"[ZERO_FLOOR={ZERO_FLOOR}] 치환된 0값: {n_total}개 (분석대상 {len(classes)}개 분류군)")
    return monthly


# ═════════════════════════════════════════════════════════════
# 2. 지표 계산 (YoY → EI → 세그먼트)
# ═════════════════════════════════════════════════════════════
def cagr(a0, a1, years):
    with np.errstate(divide="ignore", invalid="ignore"):
        return (a1 / a0) ** (1 / years) - 1


def max_drawdown(df):
    return (df / df.cummax() - 1).min()


def compute_metrics(monthly, names, classes, start, end):
    mid = end - SHORT_WINDOW
    ann = monthly[classes].groupby(monthly.index.year).sum(min_count=12)
    mk_cols = [c for c in monthly.columns if c not in EXCLUDE_CODES]
    mkt_m = monthly[mk_cols].sum(axis=1)
    mkt_ann = mkt_m.groupby(mkt_m.index.year).sum()
    mkt_ttm = mkt_m.rolling(12, min_periods=12).sum().dropna()

    # ── YoY (모든 EI의 기반) ──
    # yoy = ann.pct_change().loc[start + 1:end]
    # mkt_yoy = mkt_ann.pct_change().loc[start + 1:end]
    yoy     = ann.pct_change(fill_method=None).loc[start + 1:end]
    mkt_yoy = mkt_ann.pct_change(fill_method=None).loc[start + 1:end]

    # ── EI_t = (1+YoY_cls)/(1+YoY_mkt) × 100 ──
    ei = (1 + yoy).div(1 + mkt_yoy, axis=0) * 100
    w_years = [end - 2, end - 1, end]
    # ei_w = sum(ei.loc[y] * w for y, w in zip(w_years, EI_WEIGHTS))
    #  과거 데이터 부족시, NaN 전파 문제 해결
    w_years = [end - 2, end - 1, end]
    w = pd.Series(EI_WEIGHTS, index=w_years)
    sub = ei.loc[w_years]
    mask = sub.notna()
    ei_w = sub.mul(w, axis=0).sum(axis=0, skipna=True) / mask.mul(w, axis=0).sum(axis=0)

    m = pd.DataFrame(index=classes)
    m["약효분류명"] = names.reindex(classes)
    for y in sorted({start, mid, end - 1, end}):
        m[f"{y}"] = ann.loc[y]

    m["MS_최근(%)"] = ann.loc[end] / mkt_ann.loc[end] * 100
    m["MS_시작(%)"] = ann.loc[start] / mkt_ann.loc[start] * 100
    m["MS변화(%p)"] = m["MS_최근(%)"] - m["MS_시작(%)"]

    m["YoY_최근"] = yoy.loc[end]
    m["EI_최근"] = ei.loc[end]
    m["EI_3Y가중"] = ei_w
    m["EI>100_연수(최근3년)"] = (ei.loc[w_years] > 100).sum()
    m["EI>100_연수(전체)"] = (ei > 100).sum()

    m["CAGR_장기"] = cagr(ann.loc[start], ann.loc[end], end - start)
    m["CAGR_단기"] = cagr(ann.loc[mid], ann.loc[end], SHORT_WINDOW)
    m["격차_단기-장기(%p)"] = (m["CAGR_단기"] - m["CAGR_장기"]) * 100

    # 성장기여도(%p): 이 분류군의 시장 성장 기여도 (개별).  시장성장률차지하는 %P찾기
    # 주의: classes가 시장 전체가 아니므로 합 ≠ 시장 성장률
    m["증가액"] = ann.loc[end] - ann.loc[start]
    m["성장기여도_장기(%p)"] = m["증가액"] / mkt_ann.loc[start] * 100
    m["성장기여도_최근(%p)"] = (ann.loc[end] - ann.loc[end - 1]) / mkt_ann.loc[end - 1] * 100
    m["증가분비중(%)"] = m["증가액"] / (mkt_ann.loc[end] - mkt_ann.loc[start]) * 100

    m["음성장_연수"] = (yoy < 0).sum()
    m["YoY_최저"] = yoy.min()
    ttm = monthly[classes].rolling(12, min_periods=12).sum().dropna(how="all")
    m["MDD_TTM"] = max_drawdown(ttm)

    mk = dict(
        long=cagr(mkt_ann.loc[start], mkt_ann.loc[end], end - start),
        short=cagr(mkt_ann.loc[mid], mkt_ann.loc[end], SHORT_WINDOW),
        yoy=mkt_ann.loc[end] / mkt_ann.loc[end - 1] - 1,
        mdd=max_drawdown(mkt_ttm.to_frame())[0],
        ann=mkt_ann, yoy_series=mkt_yoy, ttm=mkt_ttm,
        ms_median=m["MS_최근(%)"].median(),
    )

    # ── 세그먼트 (EI_3Y × MS 중앙값) ──
    hi_ei, hi_ms = m["EI_3Y가중"] >= 100, m["MS_최근(%)"] >= mk["ms_median"]
    m["세그먼트"] = np.select(
        [hi_ei & ~hi_ms, hi_ei & hi_ms, ~hi_ei & hi_ms], [SEG_PROM, SEG_CORE, SEG_CASH], default=SEG_DECL)
    gap = m["CAGR_단기"] - m["CAGR_장기"]
    m["가속도"] = np.select([gap > ACCEL_TOL_CAGR, gap < -ACCEL_TOL_CAGR], ["가속", "둔화"], default="유지")
    # 성장견인군: 기여도 ≥ DRIVER_PP & CAGR > 시장
    m["성장견인군"] = (m["성장기여도_장기(%p)"] >= DRIVER_PP) & (m["CAGR_장기"] > mk["long"])
    # 최근 성장세 판정: 현재 YoY vs 5년 CAGR
    m["YoY_전년"] = yoy.loc[end - 1]
    mom = m["YoY_최근"] - m["CAGR_장기"]
    m["최근성장판정"] = np.select([m["YoY_최근"] < 0, mom > ACCEL_TOL_YOY, mom < -ACCEL_TOL_YOY],
                              ["역성장", "가속", "둔화"], default="유지")

    return m.sort_values(f"{end}", ascending=False), yoy, ei, ann, ttm, mk

# 임시
def build_growth_pool(m, mk, ctx,
                      min_ms=0.05, min_years_ei=3):
    min_cagr = mk["long"]
    median_ms = mk["ms_median"]

    ei_ok      = m["EI_3Y가중"]           >= 100
    cagr_ok    = m["CAGR_장기"]           > min_cagr
    size_ok    = m["MS_최근(%)"]          >= min_ms
    persist_ok = m["EI>100_연수(전체)"]    >= min_years_ei
    driver_ok  = m["성장견인군"].fillna(False)
    big_ok     = m["MS_최근(%)"]          >= median_ms

    strict = ei_ok & cagr_ok & size_ok & persist_ok

    # Tier 1: 성장견인 — MS 무관, 견인이면 최우선
    core = m.index[strict & driver_ok].tolist()

    # Tier 2: 유망성장 — 시장초과 + 규모 작음 + 비견인
    prom = m.index[strict & ~driver_ok & ~big_ok].tolist()

    # Tier 2b: 성숙 대형 — 시장초과 but 기여도 낮음 & 규모 큼
    mature = m.index[strict & ~driver_ok & big_ok].tolist()

    # Tier 3: 성숙견인 — 성장견인 but 최근 EI < 100 (둔화)
    decay = m.index[cagr_ok & size_ok & driver_ok & ~ei_ok].tolist()

    # Tier 4: 관찰 — EI만 충족, 나머지 미달
    watch = m.index[ei_ok & ~strict].tolist()

    return {
        "core":   core,     # 핵심주도 (성장견인)
        "prom":   prom,     # 유망성장
        "mature": mature,   # 성숙 대형
        "decay":  decay,    # 성숙견인 (둔화 중인 견인주)
        "watch":  watch,    # 관찰
        "pool":   core + prom,   # ← 최종 성장군
    }

def diagnose_drivers(m, mk, min_ms=0.05, min_years_ei=3):
    """성장견인군 중 Tier 1 탈락 사유를 진단."""
    drivers = m[m["성장견인군"].fillna(False)].copy()
    if not len(drivers):
        return pd.DataFrame()

    median_ms = mk["ms_median"]
    checks = {
        "EI≥100 미달":       drivers["EI_3Y가중"] < 100,
        "MS<0.05% 미달":     drivers["MS_최근(%)"] < min_ms,
        "MS<median":         drivers["MS_최근(%)"] < median_ms,
        "지속성 미달":        drivers["EI>100_연수(전체)"] < min_years_ei,
        "CAGR≤시장":         drivers["CAGR_장기"] <= mk["long"],
    }
    rows = []
    for c in drivers.index:
        reasons = [k for k, mask in checks.items() if mask.loc[c]]
        rows.append({
            "약효분류명": m.loc[c, "약효분류명"],
            "MS_최근(%)": m.loc[c, "MS_최근(%)"],
            "EI_3Y가중": m.loc[c, "EI_3Y가중"],
            "CAGR_장기": m.loc[c, "CAGR_장기"],
            "EI>100_연수": m.loc[c, "EI>100_연수(전체)"],
            "기여도(%p)": m.loc[c, "성장기여도_장기(%p)"],
            "탈락사유": " | ".join(reasons) if reasons else "✅ Tier1 통과",
        })
    return pd.DataFrame(rows).sort_values("기여도(%p)", ascending=False)


In [10]:

# ═════════════════════════════════════════════════════════════
# 3. 시각화 공통 도구
# ═════════════════════════════════════════════════════════════
def place_labels(ax, xs, ys, texts, fontsize=9.5, bold_idx=()):
    """겹침을 줄이는 그리디 라벨 배치. 축 스케일(log/symlog)과 무관하게 동작."""
    xs, ys = list(xs), list(ys)
    if not xs:
        return
    fr = ax.transAxes.inverted().transform(ax.transData.transform(np.column_stack([xs, ys])))
    px, py = fr[:, 0], fr[:, 1]
    ch_w, h = fontsize * 0.00135, fontsize * 0.0032
    placed = []
    cands = [(0.012, 0.012), (0.012, -0.03), (-0.012, 0.012), (-0.012, -0.03),
             (0.012, 0.045), (0.012, -0.06), (-0.012, 0.045), (-0.012, -0.06),
             (0.03, 0.0), (-0.03, 0.0), (0.03, 0.07), (-0.03, -0.07),
             (0.05, 0.03), (-0.05, -0.03)]
    for i, t in enumerate(texts):
        w = ch_w * len(t) * 1.7
        best, best_pen = None, 1e9
        for dx, dy in cands:
            left = px[i] + dx if dx > 0 else px[i] + dx - w
            rect = (left, py[i] + dy, left + w, py[i] + dy + h)
            pen = 5 if (rect[0] < 0 or rect[2] > 1 or rect[1] < 0 or rect[3] > 1) else 0
            for r in placed:
                ox = max(0, min(rect[2], r[2]) - max(rect[0], r[0]))
                oy = max(0, min(rect[3], r[3]) - max(rect[1], r[1]))
                pen += 40 * ox * oy / (w * h)
            for j in range(len(px)):
                if j != i and abs(px[j] - (rect[0] + rect[2]) / 2) < w / 2 + 0.008 \
                        and abs(py[j] - (rect[1] + rect[3]) / 2) < h / 2 + 0.008:
                    pen += 0.6
            pen += 0.02 * (abs(dx) + abs(dy))
            if pen < best_pen:
                best, best_pen = (dx, dy, rect), pen
        dx, dy, rect = best
        placed.append(rect)
        ax.annotate(t, (xs[i], ys[i]), textcoords="axes fraction",
                    xytext=(rect[0] if dx > 0 else rect[2], rect[1] + h / 2), fontsize=fontsize, zorder=6,
                    fontweight="bold" if i in bold_idx else "normal",
                    ha="left" if dx > 0 else "right", va="center",
                    arrowprops=dict(arrowstyle="-", color="#888", lw=.5, shrinkA=0, shrinkB=2),
                    bbox=dict(boxstyle="round,pad=.15", fc="white", ec="none", alpha=.85))


def _spread(vals, gap):
    order = np.argsort(vals)
    v = np.array(vals, dtype=float)[order]
    for i in range(1, len(v)):
        if v[i] - v[i - 1] < gap:
            v[i] = v[i - 1] + gap
    out = np.empty_like(v)
    out[order] = v
    return out


def bubble_size(amt, amax, base=30, k=1400):
    return base + k * (amt / amax) ** 0.8


def size_handles(amt_max, div, unit, ref=(500, 2000, 10000), base=30, k=1400):
    hs = []
    for v in [x for x in ref if x * div < amt_max]:
        hs.append(Line2D([], [], marker="o", ls="", color="#bbb",
                         ms=np.sqrt(bubble_size(v * div, amt_max, base, k)), label=f"{v:,} {unit}"))
    return hs


def save(fig, path):
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


# ═════════════════════════════════════════════════════════════
# 4. 핵심 5개 차트
# ═════════════════════════════════════════════════════════════
def fig_matrix_A(m, mk, ctx, path):
    """Matrix A — MS(log) × 3Y 가중 EI : 점유율은 작지만 시장보다 빨리 크는 군."""
    end, div, unit = ctx["end"], ctx["div"], ctx["unit"]
    x, y = m["MS_최근(%)"], m["EI_3Y가중"]
    ylo = min(y.quantile(.03), 100) - 4
    yhi = max(y.quantile(.97), 100) + 9
    yc = y.clip(ylo + 1.5, yhi - 4)
    clipped = y != yc
    xlo, xhi = x.min() * .6, x.max() * 1.8
    med = mk["ms_median"]
    amt = m[f"{end}"]
    s = bubble_size(amt, amt.max())
    col = m["세그먼트"].map(SEG_COLORS)

    fig, ax = plt.subplots(figsize=(12, 8.5))
    ax.set_xscale("log"); ax.set_xlim(xlo, xhi); ax.set_ylim(ylo, yhi)
    ax.fill_between([xlo, med], 100, yhi, color=SEG_SHADE[SEG_PROM], zorder=0)
    ax.fill_between([med, xhi], 100, yhi, color=SEG_SHADE[SEG_CORE], zorder=0)
    ax.fill_between([med, xhi], ylo, 100, color=SEG_SHADE[SEG_CASH], zorder=0)
    ax.fill_between([xlo, med], ylo, 100, color=SEG_SHADE[SEG_DECL], zorder=0)
    ax.axhline(100, color="#d62728", ls="--", lw=1.4, zorder=1)
    ax.axvline(med, color=C_MKT, ls=":", lw=1.4, zorder=1)

    ok = ~clipped
    ax.scatter(x[ok], yc[ok], s=s[ok], c=col[ok], alpha=.75, edgecolor="white", lw=.8, zorder=3)
    ax.scatter(x[clipped], yc[clipped], s=s[clipped], facecolor="none", edgecolor=col[clipped], lw=2, zorder=3)

    kw = dict(fontsize=12.5, fontweight="bold", alpha=.6, transform=ax.transAxes, zorder=2)
    ax.text(.01, .985, "유망성장군\n(점유율↓ · 성장 > 시장)", ha="left", va="top", color=SEG_COLORS[SEG_PROM], **kw)
    ax.text(.99, .985, "핵심주도군\n(점유율↑ · 성장 > 시장)", ha="right", va="top", color=SEG_COLORS[SEG_CORE], **kw)
    ax.text(.99, .015, "캐시카우\n(점유율↑ · 성장 < 시장)", ha="right", va="bottom", color="#b26a00", **kw)
    ax.text(.01, .015, "정체/쇠퇴군", ha="left", va="bottom", color=SEG_COLORS[SEG_DECL], **kw)
    ax.text(xhi, 100, " EI=100 (시장 속도)", color="#d62728", fontsize=9, va="bottom", ha="right")
    ax.text(med, ylo, f" MS 중앙값 {med:.2f}%", color=C_MKT, fontsize=9, va="bottom", ha="left")

    prom_all = m[m["세그먼트"] == SEG_PROM].index.tolist()
    lab = list(dict.fromkeys(
        prom_all
        + m[m["세그먼트"] == SEG_CORE].nlargest(6, f"{end}").index.tolist()
        + m[m["세그먼트"] == SEG_CASH].nlargest(3, f"{end}").index.tolist()))
    place_labels(ax, [x[c] for c in lab], [yc[c] for c in lab],
                 [short_name(m.loc[c, "약효분류명"]) + (f" (EI {y[c]:.0f}↑)" if clipped[c] else "") for c in lab],
                 bold_idx={i for i, c in enumerate(lab) if c in prom_all})
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}%"))
    ax.set_xlabel("최근 시장점유율 MS (%)  [로그축]", fontsize=12)
    ax.set_ylabel("최근 3개년 가중평균 EI  (100 = 시장 성장 속도)", fontsize=12)
    ax.set_title("Matrix A · 유망성장군 발굴 — 점유율(MS)(2025) × 3Y 가중 EI (2023~2025)", fontsize=15, fontweight="bold", loc="left", pad=16)
    handles = [Line2D([], [], marker="o", ls="", color=c, ms=10, label=k) for k, c in SEG_COLORS.items()]
    handles += [Line2D([], [], marker="o", ls="", mfc="none", mec="#555", mew=2, ms=10, label="축 범위 초과")]
    handles += size_handles(amt.max(), div, unit)
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(.5, -.09), ncol=4, frameon=False,
              fontsize=9.5, labelspacing=1.5, handletextpad=1.0)
    w = "·".join(f"{w:g}" for w in EI_WEIGHTS)
    fig.text(.01, -.005, f"EI = (1+YoY_분류군)/(1+YoY_시장)×100 · 3Y 가중 = {end-2}~{end}년 가중치 {w} · 버블 크기 = {end}년 규모",
             fontsize=9, color="#666")
    fig.tight_layout()
    save(fig, path)


def fig_matrix_B(m, mk, ctx, path):
    """Matrix B — 성장기여도(%p, symlog) × 장기 CAGR : 실제 시장 성장을 끈 군."""
    start, end, div, unit = ctx["start"], ctx["end"], ctx["div"], ctx["unit"]
    x, y = m["성장기여도_장기(%p)"], m["CAGR_장기"] * 100
    ml = mk["long"] * 100
    ylo = min(y.quantile(.03), ml) - 2
    yhi = max(y.quantile(.97), ml) + 6
    yc = y.clip(ylo + 1, yhi - 2.5)
    clipped = y != yc
    amt = m[f"{end}"]
    s = bubble_size(amt, amt.max())
    cat = np.select([(x >= DRIVER_PP) & (y > ml), x >= DRIVER_PP, x > 0], ["driver", "scale", "small"], default="neg")
    cmap = {"driver": C_UP, "scale": "#e6a117", "small": C_FLAT, "neg": C_DOWN}
    col = pd.Series(cat, index=m.index).map(cmap)

    fig, ax = plt.subplots(figsize=(12, 8.5))
    ax.set_xscale("symlog", linthresh=.05)
    xlo, xhi = min(-.08, x.min() * 1.6), x.max() * 1.6
    ax.set_xlim(xlo, xhi); ax.set_ylim(ylo, yhi)
    ax.fill_between([DRIVER_PP, xhi], ml, yhi, color="#e8f1fa", zorder=0)
    ax.axhline(ml, color="#d62728", ls="--", lw=1.4, zorder=1)
    ax.axvline(0, color="#777", lw=1, zorder=1)
    ax.axvline(DRIVER_PP, color=C_MKT, ls=":", lw=1.2, zorder=1)
    ok = ~clipped
    ax.scatter(x[ok], yc[ok], s=s[ok], c=col[ok], alpha=.75, edgecolor="white", lw=.8, zorder=3)
    ax.scatter(x[clipped], yc[clipped], s=s[clipped], facecolor="none", edgecolor=col[clipped], lw=2, zorder=3)

    kw = dict(fontsize=12.5, fontweight="bold", alpha=.6, transform=ax.transAxes, zorder=2)
    ax.text(.99, .985, "성장견인군\n(기여 큼 · 시장보다 빠름)", ha="right", va="top", color=C_UP, **kw)
    ax.text(.99, .015, "규모형 기여\n(기여 큼 · 시장보다 느림)", ha="right", va="bottom", color="#b26a00", **kw)
    ax.text(.01, .985, "소규모 고성장\n(빠르지만 기여 미미)", ha="left", va="top", color="#666", **kw)
    ax.text(xhi, ml, " 시장 장기 CAGR", color="#d62728", fontsize=9, va="bottom", ha="right")
    ax.text(DRIVER_PP, ylo, f" {DRIVER_PP}%p", color=C_MKT, fontsize=9, va="bottom", ha="left")

    drivers = m[m["성장견인군"]].nlargest(6, "성장기여도_장기(%p)").index.tolist()
    lab = list(dict.fromkeys(drivers + m.nlargest(6, f"{end}").index.tolist()
                             + m.nsmallest(2, "성장기여도_장기(%p)").index.tolist()
                             + [c for c in m.index[clipped] if m.loc[c, "성장기여도_장기(%p)"] >= DRIVER_PP][:3]))
    place_labels(ax, [x[c] for c in lab], [yc[c] for c in lab],
                 [short_name(m.loc[c, "약효분류명"]) + (f" (CAGR {y[c]:.0f}%↑)" if clipped[c] else "") for c in lab],
                 bold_idx={i for i, c in enumerate(lab) if c in drivers})
    ticks = [t for t in (-.1, 0, .1, .3, 1, 2, 4, 6) if xlo <= t <= xhi]
    ax.set_xticks(ticks); ax.set_xticklabels([f"{t:g}" for t in ticks])
    ax.yaxis.set_major_formatter(FuncFormatter(pct_fmt))
    ax.set_xlabel(f"성장기여도 (%p) = 분류군 증가액 ÷ {start}년 전체시장  [symlog축, 전 분류군 합 = 시장 성장률 {(mk['ann'].loc[end]/mk['ann'].loc[start]-1)*100:.0f}%]", fontsize=11)
    ax.set_ylabel(f"장기 CAGR ({start}→{end})", fontsize=12)
    ax.set_title("Matrix B · 성장견인군 발굴 — 성장기여도 × 장기 CAGR", fontsize=15, fontweight="bold", loc="left", pad=16)
    handles = [Line2D([], [], marker="o", ls="", color=cmap["driver"], ms=10, label=f"성장견인군 (기여≥{DRIVER_PP}%p & CAGR>시장)"),
               Line2D([], [], marker="o", ls="", color=cmap["scale"], ms=10, label="규모형 기여"),
               Line2D([], [], marker="o", ls="", color=cmap["small"], ms=10, label="기여 미미"),
               Line2D([], [], marker="o", ls="", color=cmap["neg"], ms=10, label="기여 음(−)")]
    handles += size_handles(amt.max(), div, unit)
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(.5, -.11), ncol=4, frameon=False,
              fontsize=9.5, labelspacing=1.5, handletextpad=1.0)
    fig.text(.01, -.005, f"버블 크기 = {end}년 규모 · 굵은 라벨 = 성장견인군 상위", fontsize=9, color="#666")
    fig.tight_layout()
    save(fig, path)


def pool_order(m):
    """POOL = 핵심주도군 + 유망성장군 전체. 그룹 내 정렬: 시장초과 연수 ↓, 3Y EI ↓ (그림 3·4·5 공통)."""
    pool = m[m["세그먼트"].isin([SEG_CORE, SEG_PROM])].copy()
    pool["_g"] = pool["세그먼트"].map({SEG_CORE: 0, SEG_PROM: 1})
    pool = pool.sort_values(["_g", "EI>100_연수(전체)", "EI_3Y가중"], ascending=[True, False, False])
    return pool.index[pool["_g"] == 0].tolist(), pool.index[pool["_g"] == 1].tolist()


def make_pool_grid(core, prom, title, ncols=4, pw=4.05, ph=2.6):
    """핵심주도군 / 유망성장군 두 블록으로 나뉜 소형 다중 격자. (code, ax, group, col) 리스트 반환."""
    rc, rp = -(-len(core) // ncols), -(-len(prom) // ncols)
    H = ph * (rc + rp) + 3.0
    fig = plt.figure(figsize=(pw * ncols, H))
    gs = fig.add_gridspec(rc + 1 + rp, ncols, height_ratios=[1] * rc + [.2] + [1] * rp, hspace=.72, wspace=.16,
                          top=1 - 1.85 / H, bottom=.75 / H, left=.045, right=.99)
    items = []
    for grp, codes, r0 in ((0, core, 0), (1, prom, rc + 1)):
        for i, c in enumerate(codes):
            items.append((c, fig.add_subplot(gs[r0 + i // ncols, i % ncols]), grp, i % ncols))
    for grp, codes, y in ((0, core, gs[0, 0].get_position(fig).y1 + .72 / H),
                          (1, prom, gs[rc, 0].get_position(fig).y0 + .10 / H)):
        seg = SEG_CORE if grp == 0 else SEG_PROM
        fig.text(.045, y, f"{seg} · {len(codes)}개", fontsize=14, fontweight="bold", color=SEG_COLORS[seg],
                 va="bottom")
        fig.add_artist(Line2D([.045, .99], [y - .004, y - .004], color=SEG_COLORS[seg], lw=2.2,
                              transform=fig.transFigure))
    fig.suptitle(title, fontsize=17, fontweight="bold", x=.045, ha="left", y=1 - .12 / H)
    return fig, items, H


def _panel_title(ax, m, c, stat, stat_color):
    ax.set_title(f"{short_name(m.loc[c, '약효분류명'], 13)} ({c})", loc="left", fontsize=11, fontweight="bold",
                 color=SEG_COLORS[m.loc[c, "세그먼트"]], pad=17)
    ax.text(0, 1.03, stat, transform=ax.transAxes, fontsize=9.3, fontweight="bold", color=stat_color, va="bottom")


def fig_yoy_heatmap(m, yoy, mk, core, prom, ctx, path):
    """③ 연도별 YoY 히트맵 (POOL 21개) — 2020→2025 동안 성장 패턴이 일관됐는가."""
    start, end, div, unit = ctx["start"], ctx["end"], ctx["div"], ctx["unit"]
    years = list(yoy.index)
    rows, labels, right, kinds = [], [], [], []

    def add(label, vals, r, kind):
        rows.append(vals); labels.append(label); right.append(r); kinds.append(kind)

    add("■ 전체 시장", list(mk["yoy_series"].loc[years].values * 100) + [mk["long"] * 100],
        f"{mk['ann'].loc[end] / div:,.0f}{unit} · MS 100%", "mkt")
    for seg, codes in ((SEG_CORE, core), (SEG_PROM, prom)):
        add(f"{seg} · {len(codes)}개", [np.nan] * (len(years) + 1), "", seg)
        for c in codes:
            add(f"{short_name(m.loc[c, '약효분류명'], 14)} ({c})",
                list(yoy.loc[years, c].values * 100) + [m.loc[c, "CAGR_장기"] * 100],
                f"{m.loc[c, f'{end}'] / div:,.1f}{unit} · MS {m.loc[c, 'MS_최근(%)']:.2f}%", "cls")
    M = np.array(rows, dtype=float)
    n = len(rows)
    cmap = plt.get_cmap("RdBu").copy(); cmap.set_bad("white")
    fig, ax = plt.subplots(figsize=(11.5, .4 * n + 2.6))
    im = ax.imshow(np.ma.masked_invalid(np.clip(M, -25, 25)), cmap=cmap,
                   norm=TwoSlopeNorm(vmin=-25, vcenter=0, vmax=25), aspect="auto")
    for i in range(n):
        for j in range(M.shape[1]):
            v = M[i, j]
            if np.isnan(v):
                continue
            ax.text(j, i, f"{v:+.1f}" if abs(v) < 10 else f"{v:+.0f}", ha="center", va="center", fontsize=9.5,
                    fontweight="bold" if v < 0 else "normal", color="white" if abs(v) > 17 else "#111")
            if v < 0 and j < len(years):
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, ec="#8b0000", lw=1.6))
    ax.axvline(len(years) - .5, color="white", lw=6)
    ax.set_xticks(range(len(years) + 1))
    ax.set_xticklabels([f"{y}\nYoY(%)" for y in years] + [f"CAGR(%)\n{start}→{end}"], fontsize=10.5)
    ax.xaxis.tick_top()
    ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=10.5)
    for i, (t, k) in enumerate(zip(ax.get_yticklabels(), kinds)):
        if k in SEG_COLORS:
            t.set_color(SEG_COLORS[k]); t.set_fontweight("bold"); t.set_fontsize(11.5)
            ax.axhline(i, color=SEG_COLORS[k], lw=1.6, xmin=0, xmax=1)
        elif k == "mkt":
            t.set_fontweight("bold")
    axr = ax.twinx(); axr.set_ylim(ax.get_ylim()); axr.set_yticks(range(n))
    axr.set_yticklabels(right, fontsize=9, color="#555"); axr.tick_params(length=0)
    ax.tick_params(length=0)
    for a in (ax, axr):
        for sp in a.spines.values():
            sp.set_visible(False)
    cb = fig.colorbar(im, ax=ax, orientation="horizontal", pad=.025, fraction=.022, shrink=.5)
    cb.set_label("◀ 감소(빨강)      0      증가(파랑) ▶   (%, ±25% 초과는 색만 포화 · 숫자는 실제값)", fontsize=9)
    ax.set_title(f"③ 연도별 YoY 히트맵 — POOL {len(core) + len(prom)}개, 성장 패턴의 일관성 (빨간 테두리 = 마이너스 성장)",
                 fontsize=14, fontweight="bold", loc="left", pad=48)
    fig.text(.01, -.005, f"YoY는 전년 값이 필요해 {years[0]}년부터 표시({start}년은 기준연도). "
             "행 순서: 그룹 내 시장초과 연수 ↓ → 3Y EI ↓ (④⑤와 동일)", fontsize=9, color="#666")
    save(fig, path)


def fig_ei_pool(m, ei, core, prom, ctx, path):
    """④ EI 추세 (POOL 21개, 소형 다중) — 시장보다 지속적으로 빠르게 성장했는가."""
    end = ctx["end"]
    years = list(ei.index)
    pool = core + prom
    allv = ei[pool].stack()
    ymax = min(max(allv.quantile(.93), 118) + 10, 150)
    ymin = max(min(allv.quantile(.04), 90) - 8, 55)
    hi_c, lo_c = ymax - 6, ymin + 6
    G, R = "#1b9e77", "#d1495b"
    fig, items, H = make_pool_grid(core, prom, "④ EI 추세 — 시장보다 지속적으로 빠르게 성장했는가? (EI 100 = 시장 속도)")
    for c, ax, grp, col in items:
        v = ei[c]
        vc = v.clip(lo_c, hi_c)
        n_up = int(m.loc[c, "EI>100_연수(전체)"])
        ax.axvspan(end - 2 - .5, end + .5, color="#eef2f6", zorder=0)
        ax.axhline(100, color="#222", ls="--", lw=1.1, zorder=1)
        ax.fill_between(years, 100, vc, where=vc >= 100, interpolate=True, color=G, alpha=.30, zorder=2)
        ax.fill_between(years, 100, vc, where=vc < 100, interpolate=True, color=R, alpha=.30, zorder=2)
        ax.plot(years, vc, color="#333", lw=1.7, zorder=3)
        for yr in years:
            up, out_hi, out_lo = v[yr] >= 100, v[yr] > hi_c, v[yr] < lo_c
            ax.scatter([yr], [vc[yr]], s=70 if (out_hi or out_lo) else 46, color=G if up else R, edgecolor="white",
                       lw=1, zorder=4, marker="^" if out_hi else ("v" if out_lo else "o"))
            ax.annotate(f"{v[yr]:.0f}", (yr, vc[yr]), xytext=(0, -13 if out_hi else (9 if (up or out_lo) else -13)),
                        textcoords="offset points", ha="center", fontsize=8.3, fontweight="bold",
                        color=G if up else R, zorder=5)
        badge = "#1b7f3b" if n_up == 5 else ("#3b8f55" if n_up == 4 else ("#777" if n_up == 3 else "#c0392b"))
        _panel_title(ax, m, c, f"시장초과 {n_up}/5년 · 3Y EI {m.loc[c, 'EI_3Y가중']:.0f}", badge)
        ax.set_ylim(ymin, ymax)
        ax.set_xlim(years[0] - .5, years[-1] + .5)
        ax.set_xticks(years); ax.set_xticklabels([f"'{y % 100:02d}" for y in years], fontsize=9)
        yt = [t for t in (60, 80, 100, 120, 140) if ymin <= t <= ymax]
        ax.set_yticks(yt); ax.tick_params(axis="y", labelsize=8.5, labelleft=(col == 0))
        ax.grid(axis="y", alpha=.2)
    handles = [plt.Rectangle((0, 0), 1, 1, fc=G, alpha=.4, label="시장보다 빠름 (EI>100)"),
               plt.Rectangle((0, 0), 1, 1, fc=R, alpha=.4, label="시장보다 느림 (EI<100)"),
               Line2D([], [], color="#222", ls="--", label="EI=100 (시장 평균 속도)"),
               plt.Rectangle((0, 0), 1, 1, fc="#eef2f6", ec="#ccc", label=f"3Y 가중 구간 {end-2}~{end} (가중치 0.2/0.3/0.5)"),
               Line2D([], [], marker="^", ls="", color="#888", label="축 범위 밖 값(▲▼, 숫자=실제값)")]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.045, 1 - .55 / H), ncol=5, frameon=False, fontsize=9.5,
               columnspacing=1.6)
    fig.text(.045, .35 / H, "EI = (1+YoY_분류군)/(1+YoY_시장)×100 · 시장초과 n/5년 = 2021~2025 중 EI≥100인 연수 · "
             "모든 패널 y축 동일 스케일", fontsize=9, color="#666")
    save(fig, path)


def fig_ttm_pool(m, ttm, yoy, core, prom, ctx, path):
    """⑤ Log-TTM 매출 궤적 (POOL 21개, 소형 다중) — 현재도 성장세가 유지/가속되고 있는가."""
    end, div, unit = ctx["end"], ctx["div"], ctx["unit"]
    pool = core + prom
    STAT = {"가속": ("▲ 가속", "#2b7bba"), "유지": ("● 유지", "#5f6b7a"),
            "둔화": ("▼ 둔화", "#e07b00"), "역성장": ("■ 역성장", "#c62828")}
    fig, items, H = make_pool_grid(core, prom, "⑤ Log-TTM 매출 궤적 — 현재도 성장세가 유지·가속되고 있는가?")
    nice = [1, 2, 3, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 10000, 20000, 30000, 50000]
    for c, ax, grp, col in items:
        sr = ttm[c] / div
        cg = m.loc[c, "CAGR_장기"]
        ref = sr.iloc[0] * (1 + cg) ** (np.arange(len(sr)) / 12)          # 5년 평균 성장 경로(시작점 연장)
        ref = pd.Series(ref, index=sr.index)
        rec = sr.iloc[-13:]
        st_txt, st_col = STAT[m.loc[c, "최근성장판정"]]
        ax.set_yscale("log")
        ax.fill_between(sr.index, ref, sr, where=sr >= ref, interpolate=True, color="#2b7bba", alpha=.16, zorder=1)
        ax.fill_between(sr.index, ref, sr, where=sr < ref, interpolate=True, color="#d1495b", alpha=.16, zorder=1)
        ax.plot(sr.index, sr, color="#9aa5b1", lw=1.8, zorder=2)
        ax.plot(ref.index, ref, color="#444", ls=":", lw=1.4, zorder=3)
        ax.plot(rec.index, rec, color=st_col, lw=3.6, solid_capstyle="round", zorder=4)
        ax.scatter([sr.index[-1]], [sr.iloc[-1]], s=46, color=st_col, edgecolor="white", zorder=5)
        lo, hi = min(sr.min(), ref.min()) * .86, max(sr.max(), ref.max()) * 1.16
        if hi / lo < 1.9:
            mid = np.sqrt(lo * hi); lo, hi = mid / 1.38, mid * 1.38
        ax.set_ylim(lo, hi)
        tk = [t for t in nice if lo * 1.02 <= t <= hi / 1.02]
        if len(tk) < 2:
            tk = [float(f"{t:.2g}") for t in np.geomspace(lo * 1.15, hi / 1.15, 2)]   # 눈금을 보기 좋은 값으로 반올림
        ax.set_yticks(tk); ax.set_yticklabels([f"{t:,.0f}" if t >= 10 else f"{t:g}" for t in tk], fontsize=8.5)
        ax.minorticks_off()
        dec = [d for d in sr.index if d.month == 12]
        ax.set_xticks(dec); ax.set_xticklabels([f"'{d.year % 100:02d}" for d in dec], fontsize=9)
        ax.set_xlim(sr.index[0], sr.index[-1] + pd.Timedelta(days=20))
        ax.grid(axis="y", alpha=.2)
        _panel_title(ax, m, c, f"{st_txt}  현재 YoY {m.loc[c, 'YoY_최근'] * 100:+.0f}% vs 5Y {cg * 100:+.0f}%", st_col)
        ax.text(.98, .04, f"{sr.iloc[-1]:,.0f}{unit}" if sr.iloc[-1] >= 10 else f"{sr.iloc[-1]:.1f}{unit}",
                transform=ax.transAxes, ha="right", va="bottom", fontsize=8.5, color=st_col, fontweight="bold")
    handles = [Line2D([], [], color="#9aa5b1", lw=2, label=f"TTM 매출 (2020.12~{end}.12)"),
               Line2D([], [], color="#5f6b7a", lw=3.6, label="최근 12개월 (색 = 판정)"),
               Line2D([], [], color="#444", ls=":", lw=1.5, label="5년 평균 성장 경로(CAGR 연장)"),
               plt.Rectangle((0, 0), 1, 1, fc="#2b7bba", alpha=.3, label="경로 상회"),
               plt.Rectangle((0, 0), 1, 1, fc="#d1495b", alpha=.3, label="경로 하회")]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.045, 1 - .55 / H), ncol=5, frameon=False, fontsize=9.5,
               columnspacing=1.6)
    tol = ACCEL_TOL_YOY * 100
    fig.text(.045, .35 / H, f"판정: 현재 YoY(={end}년 TTM) vs 5년 CAGR — ▲가속 > +{tol:.0f}%p · ●유지 ±{tol:.0f}%p · "
             f"▼둔화 < −{tol:.0f}%p · ■역성장 YoY<0 · 패널별 y축(로그) 독립 · 단위 {unit}", fontsize=9, color="#666")
    save(fig, path)

def fig_market_trajectory(m, ttm, mk, core, prom, ctx, path):
    """⑤' 시장 vs 세그먼트 TTM 지수 궤적 — 누가 시장을 이끌고 있는가.

    - 배경: 개별 분류군 궤적 (연하게)
    - 강조: 시장 전체(금액가중) + 4개 세그먼트(금액가중)
    - 지수화: 시작 TTM = 100, 로그축
    """
    end, div, unit = ctx["end"], ctx["div"], ctx["unit"]

    # ── 지수화 (금액가중) ──
    total = ttm.sum(axis=1)
    if total.iloc[0] <= 0:
        raise ValueError("시작 TTM 합이 0입니다. base 시점을 확인하세요.")
    mkt_idx = total / total.iloc[0] * 100

    idx_all = ttm.div(ttm.iloc[0]) * 100
    idx_all = idx_all.where(idx_all > 0)          # log 방어 (0/음수 → NaN)

    # ── 세그먼트별 지수 (금액가중) ──
    segs = {
        SEG_CORE: core,
        SEG_PROM: prom,
        SEG_CASH: m.index[m["세그먼트"] == SEG_CASH].tolist(),
        SEG_DECL: m.index[m["세그먼트"] == SEG_DECL].tolist(),
    }
    seg_series = {}
    for seg, cols in segs.items():
        cols = [c for c in cols if c in ttm.columns]
        if not cols:
            continue
        s = ttm[cols].sum(axis=1)
        if s.iloc[0] <= 0:
            continue
        seg_series[seg] = s / s.iloc[0] * 100

    # ── 그리기 ──
    fig, ax = plt.subplots(figsize=(14, 7.5), dpi=150)

    # (1) 개별 분류군 (배경)
    for c in idx_all.columns:
        ax.plot(idx_all.index, idx_all[c], color="#c8ced6", alpha=.15,
                lw=.7, label="_nolegend_", zorder=1)

    # (2) 시장 전체
    ax.plot(mkt_idx.index, mkt_idx, color=C_MKT, lw=3.2,
            label="시장 전체 (금액가중)", zorder=6)

    # (3) 세그먼트
    for seg, sr in seg_series.items():
        ax.plot(sr.index, sr, color=SEG_COLORS[seg], lw=2.4,
                label=seg.split(" (")[0], zorder=5)

    # (4) 기준선
    ax.axhline(100, color="#888", ls=":", lw=1.2, alpha=.85, zorder=2)

    # ── Y축: log + nice ticks ──
    ax.set_yscale("log")
    from matplotlib.ticker import ScalarFormatter, NullFormatter
    ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.yaxis.set_minor_formatter(NullFormatter())

    lo = max(idx_all.min().min(), 10) * .92
    hi = min(idx_all.max().max(), 5000) * 1.08
    nice = [20, 30, 50, 70, 100, 150, 200, 300, 500, 700,
            1000, 1500, 2000, 3000]
    tk = [t for t in nice if lo <= t <= hi]
    if len(tk) >= 2:
        ax.set_yticks(tk)
        ax.set_yticklabels([f"{t:,.0f}" for t in tk], fontsize=9.5)
    ax.set_ylim(lo, hi)
    ax.minorticks_off()

    # ── X축: 12월 눈금 ──
    dec = [d for d in ttm.index if d.month == 12]
    ax.set_xticks(dec)
    ax.set_xticklabels([f"'{d.year % 100:02d}" for d in dec], fontsize=10)
    ax.set_xlim(ttm.index[0], ttm.index[-1] + pd.Timedelta(days=20))

    ax.grid(axis="y", which="both", alpha=.22)
    ax.grid(axis="x", alpha=.12)

    # ── 라벨 ──
    ax.set_title("시장 vs 세그먼트 TTM 지수 궤적 — 누가 시장을 이끌고 있는가?",
                 fontsize=15, fontweight="bold", pad=15)
    ax.set_xlabel("연월", fontsize=11)
    ax.set_ylabel("TTM 지수 (Log, 시작=100)", fontsize=11)

    # ── 범례 ──
    handles = [
        Line2D([], [], color="#c8ced6", lw=1.5, alpha=.6,
               label=f"개별 분류군 ({len(idx_all.columns)}개, 연하게)"),
        Line2D([], [], color=C_MKT, lw=3.2, label="시장 전체 (금액가중)"),
    ]
    for seg in seg_series:
        handles.append(Line2D([], [], color=SEG_COLORS[seg], lw=2.4,
                              label=seg.split(" (")[0]))
    handles.append(Line2D([], [], color="#888", ls=":", lw=1.2, label="기준선 (100)"))
    ax.legend(handles=handles, loc="upper left", fontsize=10,
              frameon=True, framealpha=.95)

    # ── 푸터 ──
    fig.text(.01, .01,
             f"지수 = TTM_t / TTM_first × 100 · 시장/세그먼트는 금액가중 · "
             f"로그축이라 동일 비율 변화가 동일 거리 · "
             f"기간 {ttm.index[0]:%Y.%m}~{ttm.index[-1]:%Y.%m} · 단위 {unit}",
             fontsize=9, color="#666")

    save(fig, path)


def fig_segment_driver_cross(m, mk, ctx, path, label_top=12):
    """EI(최근 성장 모멘텀) × 성장기여도(시장 성장 기여) 교차 분석.

    왼쪽: EI_3Y가중(X) vs 성장기여도(Y) 산점도
          - 색 = 세그먼트, 마커 = 기여유형, 버블 = MS
          - 사분면: X=100(EI, 시장), Y=DRIVER_PP(기여도)
    오른쪽: 교차표 히트맵 — 셀에 개수 + 기여도 합
    """
    setup_font()
    end, div, unit = ctx["end"], ctx["div"], ctx["unit"]

    # ── 기여 유형 분류 ──
    m2 = m.copy()
    high = m2["성장기여도_장기(%p)"].fillna(0) >= DRIVER_PP
    m2["기여유형"] = "저기여"
    m2.loc[high, "기여유형"] = "규모기여형"
    m2.loc[m2["성장견인군"].fillna(False), "기여유형"] = "성장견인형"

    SEG_ORDER = [SEG_CORE, SEG_PROM, SEG_CASH, SEG_DECL]
    SEG_SHORT = {SEG_CORE: "핵심주도", SEG_PROM: "유망성장",
                 SEG_CASH: "캐시카우", SEG_DECL: "정체쇠퇴"}
    CT_ORDER = ["성장견인형", "규모기여형", "저기여"]
    CT_MARKER = {"성장견인형": "o", "규모기여형": "s", "저기여": "x"}

    # ── 교차표 ──
    ct_cnt = pd.crosstab(m2["세그먼트"], m2["기여유형"]) \
               .reindex(index=SEG_ORDER, columns=CT_ORDER, fill_value=0)
    ct_pp = m2.pivot_table(index="세그먼트", columns="기여유형",
                            values="성장기여도_장기(%p)", aggfunc="sum") \
              .reindex(index=SEG_ORDER, columns=CT_ORDER, fill_value=0)

    # ── 그림 ──
    fig = plt.figure(figsize=(16, 8), dpi=140)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.75, 1], wspace=.22)
    axA = fig.add_subplot(gs[0, 0])
    axB = fig.add_subplot(gs[0, 1])

    # ═══ Panel A: 산점도 (X=EI, Y=기여도) ═══
    ei_vals = m2["EI_3Y가중"]
    contrib = m2["성장기여도_장기(%p)"]
    ms = m2["MS_최근(%)"].clip(lower=0.05)
    sizes = 60 + 420 * (ms / ms.max()) ** .45

    # 축 범위 (EI 이상치 완화)
    x_lo = min(ei_vals.min() - 2, 90)
    x_hi = min(ei_vals.max() + 2, max(125, ei_vals.quantile(.97) + 2))
    y_lo = min(contrib.min() - .8, -1.2)
    y_hi = contrib.max() + .6

    # 사분면 음영 (임계: X=100, Y=DRIVER_PP)
    axA.axhspan(DRIVER_PP, y_hi, xmin=0, xmax=1,
                color="#e6f5ef", alpha=.30, zorder=0)   # 상단: 고기여
    axA.axhspan(y_lo, DRIVER_PP, xmin=0, xmax=1,
                color="#f0f0f0", alpha=.30, zorder=0)   # 하단: 저기여

    # 기준선
    axA.axvline(100, color="#444", ls="--", lw=1.5, alpha=.85, zorder=1)
    axA.axhline(DRIVER_PP, color="#444", ls=":", lw=1.5, alpha=.85, zorder=1)

    # 산점
    for seg in SEG_ORDER:
        for ct in CT_ORDER:
            mask = (m2["세그먼트"] == seg) & (m2["기여유형"] == ct)
            if not mask.any():
                continue
            axA.scatter(ei_vals[mask], contrib[mask],
                        s=sizes[mask], c=SEG_COLORS[seg],
                        marker=CT_MARKER[ct],
                        edgecolor="white", linewidth=1.1,
                        alpha=.85, zorder=3)

    # 사분면 라벨
    txt_kw = dict(fontsize=9.5, fontweight="bold", alpha=.75, zorder=2)
    axA.text(x_lo + (100 - x_lo) * .5, y_hi - (y_hi - DRIVER_PP) * .08,
             "▽ 시장 이하 성장\n+ 규모 기여\n(성숙 대형)",
             ha="center", va="top", color="#a37a00", **txt_kw)
    axA.text(100 + (x_hi - 100) * .5, y_hi - (y_hi - DRIVER_PP) * .08,
             "▲ 시장 초과 성장\n+ 규모 기여\n(성장견인)",
             ha="center", va="top", color="#1b7e57", **txt_kw)
    axA.text(x_lo + (100 - x_lo) * .5, y_lo + (DRIVER_PP - y_lo) * .08,
             "· 시장 이하\n+ 소규모",
             ha="center", va="bottom", color="#777", **txt_kw)
    axA.text(100 + (x_hi - 100) * .5, y_lo + (DRIVER_PP - y_lo) * .08,
             "· 시장 초과\n+ 소규모\n(유망 씨앗)",
             ha="center", va="bottom", color="#2b7bba", **txt_kw)

    # 상위 MS 라벨
    for c in m2.nlargest(label_top, "MS_최근(%)").index:
        axA.annotate(short_name(m2.loc[c, "약효분류명"], 10),
                     (ei_vals.loc[c], contrib.loc[c]),
                     xytext=(5, 5), textcoords="offset points",
                     fontsize=8.5, color="#333", alpha=.9, zorder=4)

    axA.set_xlim(x_lo, x_hi); axA.set_ylim(y_lo, y_hi)
    axA.set_xlabel("EI 3Y 가중평균 (100 = 시장 성장 속도)", fontsize=11)
    axA.set_ylabel("성장기여도 (%p) — 시장 성장에 대한 기여", fontsize=11)
    axA.set_title("성장 모멘텀(EI) × 시장 기여도 — 버블=MS", fontsize=13,
                  fontweight="bold", loc="left")
    axA.grid(alpha=.2)

    # 범례 2개
    leg_seg = [Line2D([], [], marker="o", ls="", markersize=10,
                       markerfacecolor=SEG_COLORS[s], markeredgecolor="white",
                       label=SEG_SHORT[s]) for s in SEG_ORDER]
    leg_ct = [Line2D([], [], marker=CT_MARKER[c], ls="", markersize=9,
                      markerfacecolor="#666", markeredgecolor="white",
                      label=c) for c in CT_ORDER]
    l1 = axA.legend(handles=leg_seg, loc="upper right", fontsize=9,
                    title="세그먼트", frameon=True, framealpha=.95)
    axA.add_artist(l1)
    axA.legend(handles=leg_ct, loc="lower left", fontsize=9,
               title="기여유형", frameon=True, framealpha=.95)

    # ═══ Panel B: 교차표 히트맵 ═══
    vals = ct_cnt.values.astype(float)
    axB.imshow(vals, cmap="YlGn", aspect="auto", vmin=0, vmax=max(vals.max(), 1))

    axB.set_xticks(range(len(CT_ORDER)))
    axB.set_xticklabels(CT_ORDER, fontsize=10)
    axB.set_yticks(range(len(SEG_ORDER)))
    axB.set_yticklabels([SEG_SHORT[s] for s in SEG_ORDER], fontsize=10)

    for i, seg in enumerate(SEG_ORDER):
        for j, ct in enumerate(CT_ORDER):
            n = int(ct_cnt.loc[seg, ct])
            pp = float(ct_pp.loc[seg, ct])
            txt = f"{n}개\n{pp:+.2f}%p" if n > 0 else "—"
            color = "white" if n > vals.max() * .6 else "#222"
            axB.text(j, i, txt, ha="center", va="center", fontsize=10.5,
                     color=color, fontweight="bold" if n > 0 else "normal")

    axB.set_title("교차표: 개수 / 기여도 합", fontsize=12,
                  fontweight="bold", loc="left")
    axB.set_xlabel("성장기여 유형", fontsize=11)
    axB.set_ylabel("세그먼트", fontsize=11)

    fig.text(.01, .005,
             f"X = EI_3Y가중 (100 = 시장 성장 속도) · "
             f"Y = 성장기여도 ≥ {DRIVER_PP}%p = 기여형 · "
             f"세그먼트는 EI_3Y × MS_중앙값 (EI 기준 재사용) · "
             f"EI 100·기여도 {DRIVER_PP} 두 임계로 사분면 분할",
             fontsize=9, color="#666")

    fig.tight_layout(rect=[0, .02, 1, 1])
    save(fig, path)
    
# ═════════════════════════════════════════════════════════════
# 5. 부록 차트 (장·단기 CAGR / 규모 상위 히트맵 / 성장 vs MDD)
# ═════════════════════════════════════════════════════════════
def figA1_cagr_matrix(m, mk, ctx, path):
    start, end, div, unit = ctx["start"], ctx["end"], ctx["div"], ctx["unit"]
    x, y = m["CAGR_장기"] * 100, m["CAGR_단기"] * 100
    ml, ms = mk["long"] * 100, mk["short"] * 100
    lo = np.floor(min(x.quantile(.03), y.quantile(.03), ml, ms)) - 2
    hi = np.ceil(max(x.quantile(.97), y.quantile(.97), ml, ms)) + 2
    xc, yc = x.clip(lo, hi), y.clip(lo, hi)
    clipped = (x != xc) | (y != yc)
    amt = m[f"{end}"]
    s = bubble_size(amt, amt.max(), 25, 1300)
    col = m["가속도"].map({"가속": C_UP, "유지": C_FLAT, "둔화": C_DOWN})
    fig, ax = plt.subplots(figsize=(12, 9))
    fr = lambda v: (v - lo) / (hi - lo)
    ax.axvspan(ml, hi, ymin=fr(ms), ymax=1, color="#e8f1fa", zorder=0)
    ax.axvspan(lo, ml, ymin=fr(ms), ymax=1, color="#eef7ee", zorder=0)
    ax.axvspan(ml, hi, ymin=0, ymax=fr(ms), color="#fdf3e4", zorder=0)
    ax.axvspan(lo, ml, ymin=0, ymax=fr(ms), color="#f8eaea", zorder=0)
    ax.plot([lo, hi], [lo, hi], "--", color="#555", lw=1, zorder=1)
    ax.axvline(ml, color=C_MKT, lw=1.2); ax.axhline(ms, color=C_MKT, lw=1.2)
    ax.scatter(xc[~clipped], yc[~clipped], s=s[~clipped], c=col[~clipped], alpha=.72, edgecolor="white", lw=.8, zorder=3)
    ax.scatter(xc[clipped], yc[clipped], s=s[clipped], facecolor="none", edgecolor=col[clipped], lw=2, zorder=3)
    kw = dict(fontsize=13, fontweight="bold", alpha=.55, zorder=2)
    ax.text(hi - .5, hi - .5, "지속성장", ha="right", va="top", color="#1f5f99", **kw)
    ax.text(lo + .5, hi - .5, "반등·신흥", ha="left", va="top", color="#2e7d32", **kw)
    ax.text(hi - .5, lo + .5, "둔화 (장기 우수 → 최근 부진)", ha="right", va="bottom", color="#b26a00", **kw)
    ax.text(lo + .5, lo + .5, "정체·쇠퇴", ha="left", va="bottom", color="#a33", **kw)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    top = list(dict.fromkeys(list(amt.nlargest(12).index) + list(m["격차_단기-장기(%p)"].nlargest(3).index)
                             + list(m["격차_단기-장기(%p)"].nsmallest(3).index)))
    place_labels(ax, [xc[c] for c in top], [yc[c] for c in top], [short_name(m.loc[c, "약효분류명"]) for c in top])
    ax.xaxis.set_major_formatter(FuncFormatter(pct_fmt)); ax.yaxis.set_major_formatter(FuncFormatter(pct_fmt))
    ax.set_xlabel(f"장기 성장: {start}→{end} CAGR", fontsize=12)
    ax.set_ylabel(f"단기 성장: {end - SHORT_WINDOW}→{end} CAGR", fontsize=12)
    ax.set_title(f"[부록] 장·단기 CAGR 매트릭스 (검은선 = 시장 장기 {ml:.1f}% / 단기 {ms:.1f}%, 대각선 위 = 가속)",
                 fontsize=14, fontweight="bold", loc="left", pad=14)
    handles = [Line2D([], [], marker="o", ls="", color=C_UP, ms=10, label="가속 (단기−장기 > +2%p)"),
               Line2D([], [], marker="o", ls="", color=C_FLAT, ms=10, label="유지"),
               Line2D([], [], marker="o", ls="", color=C_DOWN, ms=10, label="둔화 (< −2%p)")]
    handles += size_handles(amt.max(), div, unit, base=25, k=1300)
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(.5, -.09), ncol=3, frameon=False, fontsize=10,
              labelspacing=1.5)
    fig.tight_layout()
    save(fig, path)


def figA2_heatmap_top(m, yoy, mk, ctx, path, top_n=28):
    start, end, div, unit = ctx["start"], ctx["end"], ctx["div"], ctx["unit"]
    sel = m.nlargest(top_n, f"{end}").index
    years = list(yoy.index)
    rows = [list(mk["yoy_series"].loc[years].values) + [mk["long"], mk["short"]]]
    labels = ["■ 전체 시장"]
    right = [f"{mk['ann'].loc[end] / div:,.0f}"]
    for c in sel:
        rows.append(list(yoy.loc[years, c].values) + [m.loc[c, "CAGR_장기"], m.loc[c, "CAGR_단기"]])
        labels.append(f"{short_name(m.loc[c, '약효분류명'], 14)} ({c})")
        right.append(f"{m.loc[c, f'{end}'] / div:,.0f}")
    M = np.array(rows) * 100
    cols = [f"{y}\nYoY" for y in years] + [f"CAGR\n{start}→{end}", f"CAGR\n{end - SHORT_WINDOW}→{end}"]
    fig, ax = plt.subplots(figsize=(11, .36 * len(rows) + 2.4))
    im = ax.imshow(np.clip(M, -20, 20), cmap="RdBu", norm=TwoSlopeNorm(vmin=-20, vcenter=0, vmax=20), aspect="auto")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            ax.text(j, i, f"{v:+.1f}" if abs(v) < 1 else f"{v:+.0f}", ha="center", va="center", fontsize=9,
                    fontweight="bold" if v < 0 else "normal", color="white" if abs(v) > 15 else "#111")
            if v < 0 and j < len(years):
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, ec="#8b0000", lw=1.4))
    ax.axvline(len(years) - .5, color="white", lw=6); ax.axhline(.5, color="white", lw=4)
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, fontsize=10); ax.xaxis.tick_top()
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=10)
    ax.get_yticklabels()[0].set_fontweight("bold")
    ax2 = ax.twinx(); ax2.set_ylim(ax.get_ylim()); ax2.set_yticks(range(len(labels)))
    ax2.set_yticklabels(right, fontsize=9, color="#555"); ax2.set_ylabel(f"{end}년 규모 ({unit})", fontsize=9, color="#555")
    for a in (ax, ax2):
        a.tick_params(length=0)
        for sp in a.spines.values():
            sp.set_visible(False)
    cb = fig.colorbar(im, ax=ax, orientation="horizontal", pad=.03, fraction=.025, shrink=.5)
    cb.set_label("◀ 감소 (빨강)      0      증가 (파랑) ▶   (%, ±20% 포화)", fontsize=9)
    ax.set_title(f"[부록] 규모 상위 {top_n}개 연도별 YoY 히트맵 (빨간 테두리 = 마이너스 성장)",
                 fontsize=14, fontweight="bold", loc="left", pad=40)
    save(fig, path)


def figA3_risk_return(m, mk, ctx, path):
    start, end, div, unit = ctx["start"], ctx["end"], ctx["div"], ctx["unit"]
    x, y = m["MDD_TTM"] * 100, m["CAGR_장기"] * 100
    ylo = max(y.quantile(.03), -25) - 2; yhi = min(y.quantile(.97), 40) + 2
    xlo = max(x.quantile(.05), -40) - 2
    yc, xc = y.clip(ylo, yhi), x.clip(xlo, 2)
    amt = m[f"{end}"]
    s = bubble_size(amt, amt.max(), 25, 1200)
    cm = {0: "#2b7bba", 1: "#f0a30a", 2: C_DOWN}
    col = m["음성장_연수"].clip(upper=2).map(cm)
    fig, ax = plt.subplots(figsize=(12, 8.5))
    ax.axhline(mk["long"] * 100, color=C_MKT, lw=1.2); ax.axvline(mk["mdd"] * 100, color=C_MKT, lw=1.2)
    ax.scatter(xc, yc, s=s, c=col, alpha=.72, edgecolor="white", lw=.8, zorder=3)
    ax.set_xlim(xlo, 2); ax.set_ylim(ylo, yhi)
    top = list(dict.fromkeys(list(amt.nlargest(12).index) + list(m["MDD_TTM"].nsmallest(3).index)))
    place_labels(ax, [xc[c] for c in top], [yc[c] for c in top], [short_name(m.loc[c, "약효분류명"]) for c in top])
    kw = dict(fontsize=13, fontweight="bold", alpha=.5, transform=ax.transAxes)
    ax.text(.99, .985, "고성장 · 저낙폭\n(안정적)", ha="right", va="top", color="#1f5f99", **kw)
    ax.text(.01, .985, "고성장 · 고변동", ha="left", va="top", color="#b26a00", **kw)
    ax.text(.99, .015, "저성장 · 저낙폭", ha="right", va="bottom", color="#777", **kw)
    ax.text(.01, .015, "저성장 · 고낙폭 (위험)", ha="left", va="bottom", color="#a33", **kw)
    ax.xaxis.set_major_formatter(FuncFormatter(pct_fmt)); ax.yaxis.set_major_formatter(FuncFormatter(pct_fmt))
    ax.set_xlabel("최대낙폭 MDD (12개월 누적 기준, 0에 가까울수록 안정)", fontsize=12)
    ax.set_ylabel(f"장기 CAGR ({start}→{end})", fontsize=12)
    ax.set_title(f"[부록] 성장 vs 하락 위험 (검은선 = 시장: CAGR {mk['long'] * 100:.1f}%, MDD {mk['mdd'] * 100:.1f}%)",
                 fontsize=14, fontweight="bold", loc="left", pad=12)
    handles = [Line2D([], [], marker="o", ls="", color=cm[0], ms=10, label="마이너스 성장 연도 0회"),
               Line2D([], [], marker="o", ls="", color=cm[1], ms=10, label="1회"),
               Line2D([], [], marker="o", ls="", color=cm[2], ms=10, label="2회 이상")]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(.5, -.09), ncol=3, frameon=False, fontsize=10)
    fig.tight_layout()
    save(fig, path)


# ═════════════════════════════════════════════════════════════
# 6. 통합 엑셀 리포트
# ═════════════════════════════════════════════════════════════
README = [
    ("YoY", "전년 대비 성장률 = 당해 ÷ 전년 − 1. 모든 EI의 기반 지표."),
    ("EI (Evolution Index)", "(1+YoY_분류군)/(1+YoY_시장)×100. 100 = 시장과 같은 속도, 100 초과 = 시장보다 빠르게 성장."),
    ("EI_3Y가중", f"최근 3개년(끝연도 기준 −2, −1, 0년) EI의 가중평균. 가중치 {EI_WEIGHTS}."),
    ("MS", "시장점유율 = 분류군 규모 ÷ 전체 시장(제외코드 제외 전 분류군 합)."),
    ("세그먼트", "EI_3Y≥100 여부 × MS ≥ 분석대상 MS 중앙값 여부의 4분면. 유망성장군 / 핵심주도군 / 캐시카우 / 정체·쇠퇴군."),
    ("성장기여도(%p)", "분류군 증가액 ÷ 시작연도 전체시장 × 100. 전 분류군 합계 = 시장 성장률. 증가분비중(%)은 시장 증가액 중 차지하는 비율."),
    ("분석POOL", "핵심주도군 + 유망성장군 전체(그림 3·4·5 대상)."),
    ("최근성장판정", "현재 YoY(끝연도 TTM) vs 5년 CAGR: 가속(+2%p 초과) / 유지(±2%p) / 둔화(−2%p 미만) / 역성장(YoY<0)."),
    ("성장견인군", f"성장기여도(장기) ≥ {DRIVER_PP}%p 이면서 장기 CAGR이 시장 CAGR보다 높은 분류군."),
    ("CAGR 장기/단기", "시작→끝 연평균 성장률 / 최근 3년 연평균 성장률."),
    ("MDD_TTM", "12개월 누적 매출 기준 최대 낙폭(계절성 제거)."),
    ("분석대상 선정", f"결측월 없음, 최근연도 규모 ≥ {MIN_RECENT/100}억원(단위 환산), 기준연도 규모 하한 충족. 제외 사유는 '제외분류' 시트."),
    ("ZERO_FLOOR", f"분석대상의 초기 0값(censored)을 {ZERO_FLOOR}로 치환."),
    ("주의", "MS 중앙값 기준의 세그먼트는 상대 분류입니다. 소규모 분류군의 EI는 변동이 크므로 그림 3·4·5의 지속성 검증을 함께 보세요."),
]


def save_report(m, yoy, ei, ann, excluded, pool, ctx, chart_paths, outdir):
    from openpyxl.formatting.rule import ColorScaleRule
    from openpyxl.utils import get_column_letter

    start, end, div = ctx["start"], ctx["end"], ctx["div"]
    amt_cols = [c for c in m.columns if c.isdigit()] + ["증가액"]
    tbl = m.copy()
    tbl.index.name = "약효분류코드"
    for c in amt_cols:
        tbl[c] = tbl[c] / div
    tbl["분석POOL"] = ["●" if c in pool else "" for c in tbl.index]
    head = ["약효분류명", "세그먼트", "분석POOL", "최근성장판정", "성장견인군", "가속도"]
    order = head + [c for c in tbl.columns if c not in head]
    tbl = tbl[order]
    tbl.to_csv(os.path.join(outdir, "growth_metrics.csv"), encoding="utf-8-sig", float_format="%.4f")

    seg = tbl.groupby("세그먼트").agg(
        분류군수=("약효분류명", "size"), 최근규모합=(f"{end}", "sum"), MS합=("MS_최근(%)", "sum"),
        EI3Y평균=("EI_3Y가중", "mean"), 성장기여도합=("성장기여도_장기(%p)", "sum"))
    seg["상위(EI순)"] = [", ".join(short_name(n, 9) for n in
                             tbl[tbl["세그먼트"] == k].nlargest(5, "EI_3Y가중")["약효분류명"]) for k in seg.index]
    seg = seg.reindex([SEG_PROM, SEG_CORE, SEG_CASH, SEG_DECL]).dropna(how="all")

    pct_cols = ["YoY_최근", "CAGR_장기", "CAGR_단기", "YoY_최저", "MDD_TTM"]
    path = os.path.join(outdir, "growth_report.xlsx")
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        pd.DataFrame(README, columns=["항목", "정의/설명"]).to_excel(w, sheet_name="README", index=False)
        seg.to_excel(w, sheet_name="세그먼트요약")
        tbl.to_excel(w, sheet_name="지표")
        nm = m["약효분류명"]
        ei.T.assign(약효분류명=lambda d: d.index.map(nm)).to_excel(w, sheet_name="EI_연도별")
        (yoy.T).assign(약효분류명=lambda d: d.index.map(nm)).to_excel(w, sheet_name="YoY_연도별")
        (ann.div(div)).T.assign(약효분류명=lambda d: d.index.map(nm)).to_excel(w, sheet_name="연도별규모")
        excluded.rename("제외사유").to_frame().to_excel(w, sheet_name="제외분류")

        w.sheets["README"].column_dimensions["A"].width = 22
        w.sheets["README"].column_dimensions["B"].width = 120
        ws = w.sheets["세그먼트요약"]
        for L, wd in zip("ABCDEFG", (34, 10, 14, 10, 10, 14, 60)):
            ws.column_dimensions[L].width = wd
        for row in ws.iter_rows(min_row=2, min_col=3, max_col=6):
            for cell in row:
                cell.number_format = "#,##0.0"

        ws = w.sheets["지표"]
        ws.freeze_panes = "C2"
        headers = [c.value for c in ws[1]]
        for j, h in enumerate(headers, 1):
            L = get_column_letter(j)
            ws.column_dimensions[L].width = 22 if h == "약효분류명" else (32 if h == "세그먼트" else 12)
            for cell in ws[L][1:]:
                if h in pct_cols:
                    cell.number_format = "0.0%"
                elif h in amt_cols:
                    cell.number_format = "#,##0.0"
                elif h.startswith(("EI_", "MS", "성장기여도", "증가분비중", "격차")):
                    cell.number_format = "0.00" if h.startswith(("MS", "성장기여도")) else "0.0"
            if h in ("EI_최근", "EI_3Y가중"):
                ws.conditional_formatting.add(f"{L}2:{L}{ws.max_row}", ColorScaleRule(
                    start_type="num", start_value=80, start_color="B56BAE", mid_type="num", mid_value=100,
                    mid_color="FFFFFF", end_type="num", end_value=120, end_color="4CAF7A"))
            if h in ("CAGR_장기", "CAGR_단기", "YoY_최근"):
                ws.conditional_formatting.add(f"{L}2:{L}{ws.max_row}", ColorScaleRule(
                    start_type="num", start_value=-.2, start_color="F8696B", mid_type="num", mid_value=0,
                    mid_color="FFFFFF", end_type="num", end_value=.2, end_color="5A8AC6"))
        for name, fmt, (lo_, mid_, hi_) in (("EI_연도별", "0.0", (80, 100, 120)), ("YoY_연도별", "0.0%", (-.2, 0, .2))):
            ws = w.sheets[name]
            ws.freeze_panes = "B2"
            ws.column_dimensions["A"].width = 12
            n_cols = ws.max_column
            ws.column_dimensions[get_column_letter(n_cols)].width = 26
            for row in ws.iter_rows(min_row=2, min_col=2, max_col=n_cols - 1):
                for cell in row:
                    cell.number_format = fmt
            rng = f"B2:{get_column_letter(n_cols - 1)}{ws.max_row}"
            ws.conditional_formatting.add(rng, ColorScaleRule(
                start_type="num", start_value=lo_, start_color="F8696B", mid_type="num", mid_value=mid_,
                mid_color="FFFFFF", end_type="num", end_value=hi_, end_color="5A8AC6"))
        # 차트 삽입
        try:
            from openpyxl.drawing.image import Image as XLImage
            wc = w.book.create_sheet("차트")
            row = 1
            for p in chart_paths:
                img = XLImage(p)
                ratio = 1000 / img.width
                img.width, img.height = 1000, int(img.height * ratio)
                wc.add_image(img, f"A{row}")
                row += int(img.height / 20) + 3
        except Exception as e:  # pillow 없음 등
            print(f"[안내] 엑셀 차트 삽입 생략: {e}")


In [15]:

# ═════════════════════════════════════════════════════════════
# 7. main
# ═════════════════════════════════════════════════════════════
def main():
    ap = argparse.ArgumentParser()
    CSV_PATH = "/workspaces/healthcareDA/data/meft_2020-2025.csv"
    ap.add_argument("--csv", default=CSV_PATH)
    ap.add_argument("--out", default="output")
    ap.add_argument("--metric", default="사용금액_백만원", choices=list(UNITS))
    ap.add_argument("--start", type=int, default=2020)
    ap.add_argument("--end", type=int, default=2025)
    ap.add_argument("--min-base", type=float, default=MIN_BASE,
                    help="기준연도 최소 규모(백만원). 기본 300")
    ap.add_argument("--min-recent", type=float, default=MIN_RECENT,
                    help="최근연도 최소 규모(백만원). 기본 500")
    # a = ap.parse_args()
    a, _unknown = ap.parse_known_args()

    setup_font()
    os.makedirs(a.out, exist_ok=True)
    div, unit = UNITS[a.metric]
    ctx = dict(start=a.start, end=a.end, div=div, unit=unit)

    monthly, names = load(a.csv, a.metric)
    classes, excluded = select_classes(monthly, a.start, a.end, a.min_base, a.min_recent)
    monthly = apply_zero_floor(monthly, classes)
    m, yoy, ei, ann, ttm, mk = compute_metrics(monthly, names, classes, a.start, a.end)
    core, prom = pool_order(m)
    pool = core + prom

    # fig7 segment-driver cross 
    # 핵심주도군인데 성장견인형이 아닌 것들 (성숙 대형 후보)
    suspect = m[(m["세그먼트"] == SEG_CORE) & (~m["성장견인군"].fillna(False))]
    if len(suspect):
        print("\n[핵심주도군 ∩ 비(성장견인형) — 성숙 대형 또는 최근 약화]")
        cols = ["약효분류명", "MS_최근(%)", "EI_3Y가중", "CAGR_장기", "성장기여도_장기(%p)"]
        print(suspect[cols].sort_values("성장기여도_장기(%p)", ascending=False).round(3).to_string())

    diag = diagnose_drivers(m, mk)
    print("\n[성장견인군 Tier1 진단]")
    print(diag.round(3).to_string())

    P = lambda f: os.path.join(a.out, f)
    charts = [P("1_matrix_A_promising.png"), P("2_matrix_B_drivers.png"), P("3_yoy_heatmap_pool.png"),
              P("4_ei_trend_pool.png"), P("5_log_ttm_pool.png"), 
              P("A1_cagr_short_vs_long.png"), P("A2_yoy_heatmap_top.png"), P("A3_growth_vs_mdd.png")
              ,P("6_market_trajectory.png"),P("7_segment_driver_cross.png")]
    fig_matrix_A(m, mk, ctx, charts[0])
    fig_matrix_B(m, mk, ctx, charts[1])
    fig_yoy_heatmap(m, yoy, mk, core, prom, ctx, charts[2])
    fig_ei_pool(m, ei, core, prom, ctx, charts[3])
    fig_ttm_pool(m, ttm, yoy, core, prom, ctx, charts[4])
    
    figA1_cagr_matrix(m, mk, ctx, charts[5])
    figA2_heatmap_top(m, yoy, mk, ctx, charts[6])
    figA3_risk_return(m, mk, ctx, charts[7])
    fig_market_trajectory(m, ttm, mk, core, prom, ctx, charts[8]) 
    fig_segment_driver_cross(m, mk, ctx, charts[9]) 

    save_report(m, yoy, ei, ann, excluded, pool, ctx, charts[:5], a.out)

    # 콘솔 요약
    print(f"분석 대상 {len(classes)}개 (제외 {len(excluded)}개) | MS 중앙값 {mk['ms_median']:.3f}%")
    print(f"시장  장기CAGR {mk['long']:.1%} | 단기CAGR {mk['short']:.1%} | 최근YoY {mk['yoy']:.1%} | MDD {mk['mdd']:.1%}")
    print("\n[세그먼트 분포]\n", m["세그먼트"].value_counts().to_string())
    cols = ["약효분류명", "MS_최근(%)", "EI_3Y가중", "EI>100_연수(전체)", "YoY_최근", "CAGR_장기", "최근성장판정"]
    print(f"\n[분석 POOL {len(pool)}개 = 핵심주도 {len(core)} + 유망성장 {len(prom)}]\n", m.loc[pool, cols].round(3).to_string())
    print("\n[최근 성장세 판정]", m.loc[pool, "최근성장판정"].value_counts().to_dict())
    print("\n[성장견인군 (기여도 순)]\n", m[m["성장견인군"]].nlargest(6, "성장기여도_장기(%p)")[
        ["약효분류명", "성장기여도_장기(%p)", "증분비중" if False else "증가분비중(%)", "CAGR_장기"]].round(3).to_string())
    


if __name__ == "__main__":
    main()


[ZERO_FLOOR=0.1] 치환된 0값: 0개 (분석대상 104개 분류군)

[핵심주도군 ∩ 비(성장견인형) — 성숙 대형 또는 최근 약화]
         약효분류명  MS_최근(%)  EI_3Y가중  CAGR_장기  성장기여도_장기(%p)
219  기타의 순환계용약     2.907  100.329    0.054         0.980
634      혈액제제류     1.057  102.717    0.066         0.419
115    각성제,흥분제     0.265  109.674    0.247         0.257
235    최토제,진토제     0.384  107.844    0.103         0.216

[성장견인군 Tier1 진단]
                   약효분류명  MS_최근(%)  EI_3Y가중  CAGR_장기  EI>100_연수  기여도(%p)                탈락사유
0                 항악성종양제    11.682  107.007    0.136          5    7.958          ✅ Tier1 통과
1                 동맥경화용제    11.114  101.476    0.101          5    6.152          ✅ Tier1 통과
2              기타의 화학요법제     4.535  135.821    0.149          1    3.281              지속성 미달
4   자격료법제(비특이성면역원제제를 포함)     2.894  103.923    0.137          5    1.984          ✅ Tier1 통과
3                   안과용제     3.162   99.675    0.078          3    1.427           EI≥100 미달
5                  진해거담제     1.793   93.235    0.135      